In [30]:
import pandas as pd
import os 

In [31]:
usuario = os.getlogin() 

In [32]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [33]:
import pandas as pd

sheet_id = "1maRNGDuU9rEFWZLgMdhJS1waAnJxl6ENntm-nyD0tq8"
gid = "1765182479"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

base_an = pd.read_csv(url)



In [34]:
col = ["clues_imb"]
base = base[col]

In [35]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [36]:
colum =['clues_imb', 'entidad','consultorio','pregunta']
base_an = base_an[colum]    

In [37]:
b = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UM_IMB_SUS.xlsx",sheet_name="Hoja2")

In [38]:
b = b.drop(index=[0, 2, 5,3,1])

In [39]:
b = b.drop(columns=['Unnamed: 1'])

In [40]:
b = b.rename(columns={
   'CLUES' : 'preguntas',
})

In [41]:
b.columns

Index(['preguntas'], dtype='object')

In [42]:
base

,clues_imb,entidad
0,BCIMB000051,BAJA CALIFORNIA
1,BCIMB000092,BAJA CALIFORNIA
2,BCIMB000162,BAJA CALIFORNIA
3,BCIMB000186,BAJA CALIFORNIA
4,BCIMB000191,BAJA CALIFORNIA
...,...,...
2822,GRIMB010244,GUERRERO
2823,GRIMB010256,GUERRERO
2824,GRIMB010565,GUERRERO
2825,SRIMB001071,SONORA


In [43]:
base_an.head()

,clues_imb,entidad,consultorio,pregunta
0,BSIMB000025,BAJA CALIFORNIA SUR,NaN,internet
1,BSIMB000025,BAJA CALIFORNIA SUR,1.0,turno_consultorio_1
2,BCIMB000092,BAJA CALIFORNIA,NaN,internet
3,BSIMB000136,BAJA CALIFORNIA SUR,NaN,internet
4,BSIMB000136,BAJA CALIFORNIA SUR,1.0,turno_consultorio_1


In [44]:
base_an = base_an[
    ~base_an["pregunta"].str.contains("internet|turno_consultorio", case=False, na=False)
]

In [45]:
base_conteo =base["clues_imb"].unique()
base_conteo

array(['BCIMB000051', 'BCIMB000092', 'BCIMB000162', ..., 'GRIMB010565',
       'SRIMB001071', 'TCIMB000704'], shape=(2191,), dtype=object)

In [46]:
total_unidades = base["clues_imb"].nunique()

respondieron = base_an["clues_imb"].nunique()

sin_responder = total_unidades - respondieron

avance_general = round(respondieron / total_unidades * 100, 1)

## tablas

In [47]:
# TABLA 1 - AVANCE POR ENTIDAD

# Unidades esperadas
esperadas = (
    base
    .groupby("entidad")
    .size()
    .reset_index(name="esperadas")
)

# Unidades que respondieron
contestadas = (
    base_an[["entidad","clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")
    .size()
    .reset_index(name="respondieron")
)

tabla_entidades = (
    esperadas
    .merge(contestadas,
           on="entidad",
           how="left")
)

tabla_entidades["respondieron"] = (
    tabla_entidades["respondieron"]
    .fillna(0)
    .astype(int)
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondieron"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)


In [48]:
tabla_entidades

,entidad,esperadas,respondieron,porcentaje
15,SAN LUIS POTOSI,33,15,45.5
14,QUINTANA ROO,165,22,13.3
20,VERACRUZ DE IGNACIO DE LA LLAVE,128,9,7.0
21,ZACATECAS,23,1,4.3
0,BAJA CALIFORNIA,92,4,4.3
3,CHIAPAS,25,1,4.0
9,MICHOACAN DE OCAMPO,43,1,2.3
2,CAMPECHE,45,1,2.2
19,TAMAULIPAS,210,4,1.9
1,BAJA CALIFORNIA SUR,70,1,1.4


In [49]:
# TABLA 2 - COMPLETITUD POR UNIDAD

# Número de preguntas del formulario
n_preguntas = len(b)

# Número de consultorios por unidad
consultorios = (
    base_an
    .groupby(["clues_imb","entidad"],as_index=False)
    ["consultorio"]
    .max()
)

consultorios.rename(
    columns={"consultorio":"consultorios"},
    inplace=True
)

# Preguntas respondidas
respondidas = (
    base_an
    .groupby(["clues_imb","entidad"])
    .size()
    .reset_index(name="respondidas")
)

tabla_unidades = consultorios.merge(
    respondidas,
    on=["clues_imb","entidad"]
)

tabla_unidades["esperadas"] = (
    tabla_unidades["consultorios"]
    * n_preguntas
)

tabla_unidades["porcentaje"] = (
    tabla_unidades["respondidas"]
    / tabla_unidades["esperadas"]
    *100
).round(1)

tabla_unidades = (
    tabla_unidades
    .rename(columns={"clues_imb":"clues"})
    .sort_values("porcentaje")
)

In [50]:
tabla_unidades

,clues,entidad,consultorios,respondidas,esperadas,porcentaje
7,MCIMB003941,MEXICO,1.0,1,44.0,2.3
5,CCIMB000102,CAMPECHE,1.0,1,44.0,2.3
24,QRIMB001536,QUINTANA ROO,1.0,1,44.0,2.3
8,MNIMB001316,MICHOACAN DE OCAMPO,1.0,2,44.0,4.5
27,QRIMB001664,QUINTANA ROO,1.0,4,44.0,9.1
...,...,...,...,...,...,...
59,VZIMB007942,VERACRUZ DE IGNACIO DE LA LLAVE,4.0,176,176.0,100.0
54,VZIMB002885,VERACRUZ DE IGNACIO DE LA LLAVE,1.0,44,44.0,100.0
58,VZIMB007616,VERACRUZ DE IGNACIO DE LA LLAVE,1.0,44,44.0,100.0
57,VZIMB007604,VERACRUZ DE IGNACIO DE LA LLAVE,1.0,44,44.0,100.0


In [51]:
# LISTAS PARA JINJA2
entidades = tabla_entidades.to_dict(orient="records")
unidades = tabla_unidades.to_dict(orient="records")

In [52]:
tabla_entidades = (
    tabla_unidades
    .groupby("entidad", as_index=False)
    .agg(
        consultorios=("consultorios","sum"),
        respondidas=("respondidas","sum"),
        esperadas=("esperadas","sum"),
        unidades=("clues","count")
    )
)

tabla_entidades["porcentaje"] = (
    tabla_entidades["respondidas"]
    / tabla_entidades["esperadas"]
    *100
).round(1)

tabla_entidades = tabla_entidades.sort_values(
    "porcentaje",
    ascending=False
)

In [53]:
tabla_entidades

,entidad,consultorios,respondidas,esperadas,unidades,porcentaje
11,ZACATECAS,6.0,262,264.0,1,99.2
10,VERACRUZ DE IGNACIO DE LA LLAVE,29.0,1256,1276.0,9,98.4
7,QUINTANA ROO,30.0,1205,1320.0,22,91.3
8,SAN LUIS POTOSI,29.0,1124,1276.0,15,88.1
0,BAJA CALIFORNIA,5.0,193,220.0,4,87.7
9,TAMAULIPAS,4.0,143,176.0,4,81.2
6,NAYARIT,1.0,7,44.0,1,15.9
3,CHIAPAS,1.0,5,44.0,1,11.4
1,BAJA CALIFORNIA SUR,1.0,4,44.0,1,9.1
5,MICHOACAN DE OCAMPO,1.0,2,44.0,1,4.5


In [54]:
# Total de unidades por entidad (base completa)
total_por_entidad = (
    base
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="total_unidades")
)

# Unidades que han respondido al menos una pregunta
respondieron_por_entidad = (
    base_an[["entidad", "clues_imb"]]
    .drop_duplicates()
    .groupby("entidad")["clues_imb"]
    .nunique()
    .reset_index(name="unidades_respondieron")
)

tabla_avance = total_por_entidad.merge(
    respondieron_por_entidad,
    on="entidad",
    how="left"
)

tabla_avance["unidades_respondieron"] = (
    tabla_avance["unidades_respondieron"].fillna(0).astype(int)
)

tabla_avance["porcentaje"] = (
    tabla_avance["unidades_respondieron"]
    / tabla_avance["total_unidades"]
    * 100
).round(1)

tabla_avance = tabla_avance.sort_values("porcentaje", ascending=False)

In [55]:
tabla_avance

,entidad,total_unidades,unidades_respondieron,porcentaje
15,SAN LUIS POTOSI,28,15,53.6
14,QUINTANA ROO,133,22,16.5
20,VERACRUZ DE IGNACIO DE LA LLAVE,74,9,12.2
0,BAJA CALIFORNIA,73,4,5.5
21,ZACATECAS,19,1,5.3
3,CHIAPAS,24,1,4.2
9,MICHOACAN DE OCAMPO,27,1,3.7
2,CAMPECHE,31,1,3.2
19,TAMAULIPAS,150,4,2.7
1,BAJA CALIFORNIA SUR,51,1,2.0


## graficas

In [56]:
import plotly.express as px

tabla = tabla_avance.sort_values("porcentaje", ascending=False).copy()

# Semáforo de colores
tabla["color"] = tabla["porcentaje"].apply(
    lambda p:
        "#0D5D2A" if p == 100 else
        "#88A91E" if p >= 80 else
        "#F1D54A" if p >= 50 else
        "#D41111"
)

fig = px.bar(
    tabla,
    x="entidad",
    y="porcentaje",
    text="porcentaje",
    title="Avance por entidad sobre unidades"
)

fig.update_traces(
    marker_color=tabla["color"],
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Avance: %{y:.1f}%<extra></extra>"
)

fig.update_yaxes(showticklabels=False)

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    template="plotly_white",
    width=1200,
    height=700
)

fig.show()

In [57]:
import plotly.graph_objects as go

# Ordenar de menor a mayor para efecto cascada
df_plot = tabla_entidades.sort_values("porcentaje", ascending=True)

# Semáforo de colores
colores = [
    "#0D5D2A" if p >= 95 else
    "#88A91E" if p >= 80 else
    "#F1D54A" if p >= 60 else
    "#D41111"
    for p in df_plot["porcentaje"]
]

fig_cascada = go.Figure()

fig_cascada.add_trace(
    go.Bar(
        x=df_plot["porcentaje"],
        y=df_plot["entidad"],
        orientation="h",
        marker=dict(color=colores),
        text=[f"{p:.1f}%" for p in df_plot["porcentaje"]],
        textposition="outside",
        textfont=dict(size=11),
        hovertemplate="<b>%{y}</b><br>Porcentaje: %{x:.1f}%<extra></extra>",
    )
)

fig_cascada.update_layout(
    title=dict(
        text="Gráfica de llenado de consultorios",
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title="",
        range=[0, 115],
        ticksuffix="%",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    yaxis=dict(
        title="",
        automargin=True
    ),
    plot_bgcolor="white",
    height=500,
    margin=dict(l=20, r=60, t=60, b=40)
)

fig_cascada.show()

In [58]:
from pathlib import Path
import json
from datetime import datetime
from plotly.utils import PlotlyJSONEncoder

base_dir = Path.cwd()
salida_data = base_dir / "data.js"
html_origen = base_dir / "reporte_interactivo.html"
index_destino = base_dir / "index.html"

figures = [
    {"id": "fig_avance", "figure": fig.to_plotly_json()},
]

if "fig_cascada" in globals():
    figures.append({"id": "fig_cascada", "figure": fig_cascada.to_plotly_json()})
else:
    print("Aviso: fig_cascada no existe todavia; se exporta solo fig_avance.")

payload = {
    "title": "Tablero IMSS Bienestar",
    "updated_at": datetime.now().isoformat(timespec="seconds"),
    "tables": {
        "tabla_avance": {
            "columns": list(tabla_avance.columns),
            "rows": tabla_avance.to_dict(orient="records"),
            "excel": "tabla_avance.xlsx",
        },
        "tabla_entidades": {
            "columns": list(tabla_entidades.columns),
            "rows": tabla_entidades.to_dict(orient="records"),
            "excel": "tabla_entidades.xlsx",
        },
        "tabla_unidades": {
            "columns": list(tabla_unidades.columns),
            "rows": tabla_unidades.to_dict(orient="records"),
            "excel": "tabla_unidades.xlsx",
        },
    },
    "figures": figures,
}

contenido_js = "window.DASHBOARD_DATA = " + json.dumps(payload, ensure_ascii=False, cls=PlotlyJSONEncoder) + ";\n"
salida_data.write_text(contenido_js, encoding="utf-8")

# Si no existe el HTML origen, se crea desde index.html para que lo puedas editar
if not html_origen.exists() and index_destino.exists():
    html_origen.write_text(index_destino.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Se creo plantilla HTML: {html_origen}")

if html_origen.exists():
    index_destino.write_text(html_origen.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"Index actualizado desde: {html_origen}")
else:
    print(f"No se encontro {html_origen.name}; index.html no se pudo actualizar.")

print(f"Datos exportados en: {salida_data}")
print("Ejecuta esta celda tras modificar graficas o HTML para refrescar el tablero.")

Index actualizado desde: c:\Users\jose.valdez\Downloads\nuevo-inf\informe--de-ang\reporte_interactivo.html
Datos exportados en: c:\Users\jose.valdez\Downloads\nuevo-inf\informe--de-ang\data.js
Ejecuta esta celda tras modificar graficas o HTML para refrescar el tablero.
